#Import required libraries

In [ ]:
import xml.etree.ElementTree as ET
from glob import glob
import pandas as pd
from shutil import copyfile
import numpy as np
import cv2
from google.colab.patches import cv2_imshow
import numpy as np
import os
from lxml import etree
import xml.etree.ElementTree as ET1
import shutil

#Provide necessary paths 

In [ ]:
path0 = r"C:\Users\KIIT\Downloads\Dental_new"#Path of new XML
path1 = r"C:\Users\KIIT\Downloads\Folder-20210130T092209Z-001\Folder"#Path for X-ray images for cropping
path2 = r"C:\Users\KIIT\Downloads\Output XML-20210130T091512Z-001\Output XML\\"#Path of old XML

#Transforming the xml into proper format

In [ ]:
def prettyPrintXml(xmlFilePathToPrettyPrint):
    assert xmlFilePathToPrettyPrint is not None
    parser = etree.XMLParser(resolve_entities=False, strip_cdata=False)
    document = etree.parse(xmlFilePathToPrettyPrint, parser)
    document.write(xmlFilePathToPrettyPrint, pretty_print=True, encoding='utf-8')

#Cropping X-ray Images

In [ ]:
for file in os.listdir(path2):
    sub_path = os.path.join(path0,os.path.splitext(file)[0])
    #print(sub_path)
    nm = os.path.splitext(file)[0]
    tree = ET1.parse(path2+file)
    root_xml = tree.getroot()
    coordinates = []
    final = []
    for x,y in zip(root_xml.findall('./annotations/element/polygon/path/element/x'),root_xml.findall('./annotations/element/polygon/path/element/y')):
        value = (int(float(x.text)),int(float(y.text)))
        final.append(list(value))
    for name in root_xml.findall('./annotations/element/name'):
        coordinates.append(name.text)
    #print(file)
    #print(coordinates,end='')
    annotations = glob(path2+file)
    df = []
    cnt = 0
    len_list = []
    for file in annotations:
        prev_filename = file.split('/')[-1].split('.')[0] + '.jpg'
    filename = str(cnt) + '.jpg'
    row = []
    for node in tree.getroot().findall('./annotations/element'):
        label = node.find('name').text
        x = node.findall(r'polygon/path/element/x')
        y = node.findall(r'polygon/path/element/y')
        if len(x) != 0:
            len_list.append(len(x))
        df.append(row)
        cnt += 1
    cum = [])
    for i in range(len(len_list)):
        if i==0:
            cum.append(len_list[i])
        else:
            cum.append(cum[i-1]+len_list[i])

    #print(cum)
    root = ET.Element("annotation") 
    folder = ET.SubElement(root, "folder") 
    filename = ET.SubElement(root, "filename")
    path  = ET.SubElement(root, "path")
    source = ET.SubElement(root, "source")
    database = ET.SubElement(source, "database")
    size = ET.SubElement(root, "size")
    width = ET.SubElement(size, "width")
    height = ET.SubElement(size, "height")
    depth = ET.SubElement(size, "depth")
    segmented = ET.SubElement(root, "segmented")
    database.text = "Unknown"
    segmented.text = str(0)
    filename.text = str(nm)+".jpg"
    height.text = root_xml.find('height')
    width.text = root_xml.find('width')
    depth.text = str(3)
    new_path = os.path.join(path1,str(nm)+".jpg")
    
    for i in range(len(cum)):
        if (i == 0):
            points = np.array(([final[i:i+cum[i]]]))
        else:
            points = np.array(([final[cum[i-1]:cum[i]]]))
        img = cv2.imread(new_path)
        mask = np.zeros(img.shape[0:2], dtype=np.uint8)
        cv2.drawContours(mask, [points], -1, (255, 255, 255), -1, cv2.LINE_AA)
        res = cv2.bitwise_and(img,img,mask = mask)
        rect = cv2.boundingRect(points) 
        cropped = res[rect[1]: rect[1] + rect[3], rect[0]: rect[0] + rect[3]]
        #print(rect[1],rect[1] + rect[3], rect[0], rect[0] + rect[3])
 
        object = ET.SubElement(root, "object")
        ET.SubElement(object, "name").text = coordinates[i]
        ET.SubElement(object, "pose").text = "Unspecified"
        ET.SubElement(object, "truncated").text = str(0)
        ET.SubElement(object, "difficult").text = str(0)
        bndbox = ET.SubElement(object, "bndbox")
        ET.SubElement(bndbox, "xmin").text = str(rect[0])
        ET.SubElement(bndbox, "ymin").text = str(rect[1])
        ET.SubElement(bndbox, "xmax").text = str(rect[0] + rect[3])
        ET.SubElement(bndbox, "ymax").text = str(rect[1] + rect[3])
        #print(rect[1],rect[1] + rect[3], rect[0], rect[0] + rect[3])
        wbg = np.ones_like(img, np.uint8)*255
        cv2.bitwise_not(wbg,wbg, mask=mask)
        dst = wbg+res
        save_path = os.path.join(sub_path, str(nm)+'_'+coordinates[i]+'.jpg') 
        #print(save_path)
        #saving the cropped image in rrespective folder of an x-ray image.
        cv2.imwrite(save_path,cropped)
    tree = ET.ElementTree(root) 
    #print(path0+str(nm)+".xml")
    save_path = os.path.join(path0,str(nm)+".xml")
    #saving the converted xml file with four coordinates.
    tree.write(save_path)
    prettyPrintXml(save_path)
    

#Converting cropped images into one CSV

In [ ]:
#Path for cropped images with all classes in one folder
path60 = r"C:\Users\KIIT\Downloads\All_dental-20210204T174836Z-001\All_dental\\"
#Path for re-arranging the images into different labelwise folders
path61 = r"C:\Users\KIIT\Downloads\cropped\train"

In [ ]:
_list = ['T','H','U']
for files in os.listdir(path60):
    #print(files)
    c=c+1
    for file in os.listdir(os.path.join(path60,files)):
      name = os.path.splitext(os.path.basename(file))[0][0]
      if (name) in _list:
        dest = os.path.join("C:\Users\KIIT\Downloads\cropped\train",name)
        shutil.copy(os.path.join(path60,files,file),dest)

In [ ]:
#path for cropped images
path61 = r"C:\Users\KIIT\Downloads\ALL"

In [ ]:
file_names=[]
for files in os.listdir(path61):
    if files.endswith('jpg'):
        file_names.append(files)

In [ ]:
label = []
for files in os.listdir(path61):
    if files.endswith('jpg'):
        label.append(os.path.splitext(os.path.basename(files))[0][0])

In [ ]:
df = pd.DataFrame(list(zip(file_names,label )),columns =['id','Label']) 

In [ ]:
df.to_csv('train.csv') 